In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from datasets.dataset import OxfordIIITPetTrainDataset, OxfordIIITPetTestDataset
from models.model_torch import train_model, save_checkpoint, load_checkpoint, UNet
from utils.util import plot_segmentation

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
train_dataset = OxfordIIITPetTrainDataset()
test_dataset = OxfordIIITPetTestDataset()
batch_size = 32

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(train_dataset, [.8,.2], generator=generator)
first_1000_train_ds = Subset(train_dataset, range(1000))

train_dataloader = DataLoader(first_1000_train_ds, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(len(first_1000_train_ds), len(val_dataset))

### Explore datasets

In [ ]:
for batch , (x, y) in enumerate(train_dataloader):
    print(x.shape, y)

In [ ]:
image, target = train_dataset[0]
original_idx = train_dataset.indices[0]
image_path = train_dataset.dataset._images[original_idx] 
filename = image_path.name  
print(filename)
print(type(image), type(target), image.shape, target)
breed_name = train_dataset.dataset.classes[target]
print(breed_name)
# Define inverse normalization to bring pixel values back to [0, 1] range for plotting
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

plot_segmentation(image,target)

### Training model

In [ ]:
num_classes = 37 # 37 breeds of dog and cat
learning_rate = 1e-3
weight_decay = 1e-4
epochs = 3 # use small epochs for transfer learning

model = UNet(num_classes).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs, # number of epochs
    eta_min=1e-6
)
loss_fn = nn.CrossEntropyLoss()

### Refresh cache on mps

In [ ]:
import torch
import gc

# Run this inside your loop or after heavy operations
torch.mps.empty_cache()
gc.collect()

In [ ]:
epochs = 1 # use small epochs for transfer learning
history = train_model(
    epochs, model, train_dataloader, val_dataloader, loss_fn, optimizer, scheduler,
    batch_size, num_classes, device,
    checkpoint_path="save/latest.pth",
    best_checkpoint_path="save/best.pth",
    resume=True,
)
print("train_model", history)


In [ ]:
checkpoint_path = "save/unet.pth"
model = UNet(num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1, eta_min=1e-6)
checkpoint = load_checkpoint(checkpoint_path, model, optimizer, scheduler, map_location=device)
print(f"Checkpoint from epoch {checkpoint['epoch']} successfully loaded.")

In [ ]:
checkpoint_path = "save/unet.pth"
save_checkpoint(
    checkpoint_path,
    epoch=len(history["train_loss"]) - 1,
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    metrics={"best_val_correct": max(history["val_correct"], default=0.0)},
)
print(f"Checkpoint successfully saved to {checkpoint_path}")

### Visualize prediction

In [ ]:
image, target = test_dataset[5]
plot_segmentation(image,target)

# pred = model(image)
# plot_segmentation(image,pred)
# print("target shape:", pred.shape)